# Fundamentals 03 - Agent API

**Historia:** ya tienes tools. Ahora construyes agentes que deciden cu?ndo usarlas.

Este notebook materializa dos rutas de `lab.agent(...)`:

1. **Agente determinista** con `engine="python-direct"`: local, reproducible y sin proveedor externo.
2. **Agente LM** con `runtime(provider="auto")`: el mismo contrato se ejecuta con el backend disponible en el ambiente.

La intenci?n did?ctica es separar la API de Agentic Systems del backend real: el contrato, la policy, la validaci?n y `human_result(...)` son iguales en ambas rutas.

In [ ]:
import agentic_systems as lab

PRETTY = False

scheduler = lab.scheduler(
    timeout_s=60,
    max_retries=0,
    max_tool_calls=4,
    max_turns=8,
)

deterministic_runtime = lab.runtime(
    provider="python-direct",
    model="local-python",
    region="local",
    scheduler=scheduler,
)

lm_runtime = lab.runtime(
    provider="auto",
    scheduler=scheduler,
)

lab.show({
    "deterministic_runtime": deterministic_runtime.describe(),
    "lm_runtime_auto_resolution": lm_runtime.describe(),
    "pretty_human_results": PRETTY,
}, title="Agent runtimes")

## Problema default de fundamentals

Todos los notebooks de `tutorials/` usan este mismo problema para comparar la API sin cambiar de caso cada vez:

```text
Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final
```

Resultado esperado: **42**.

In [ ]:
USER_PROMPT = """Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final""".strip()

# La estructura solicitada por el usuario se materializa como datos simples.
# No es un parser ni una respuesta precocinada: sólo representa la sección `Dime:`.
REQUESTED_OUTPUTS = ["procedimiento", "resultado_final"]

lab.show({
    "prompt_usuario": USER_PROMPT,
    "salidas_solicitadas": REQUESTED_OUTPUTS,
}, title="Problema default · visible")


## 1) Definir las tools del problema default

Un agente, cuatro tools. El contrato va a exigir que el agente use todas para dejar evidencia paso a paso.

In [ ]:
@lab.tool
def sumar(a: int, b: int) -> dict:
    """Suma dos números."""
    return {"operation": "sumar", "result": a + b, "explanation": f"{a} + {b} = {a + b}"}


@lab.tool
def restar(a: int, b: int) -> dict:
    """Resta dos números."""
    return {"operation": "restar", "result": a - b, "explanation": f"{a} - {b} = {a - b}"}


@lab.tool
def multiplicar(a: int, b: int) -> dict:
    """Multiplica dos números."""
    return {"operation": "multiplicar", "result": a * b, "explanation": f"{a} × {b} = {a * b}"}


@lab.tool
def dividir(a: int, b: int) -> dict:
    """Divide dos números."""
    if b == 0:
        raise ValueError("No se puede dividir entre cero.")
    value = a / b
    result = int(value) if value.is_integer() else value
    return {"operation": "dividir", "result": result, "explanation": f"{a} ÷ {b} = {result}"}


tools = [sumar, restar, multiplicar, dividir]
lab.show({"tools": [tool.name for tool in tools]})

## 2) Definir contrato y policy

El contrato declara qué debe pasar; la policy limita cómo puede pasar.

In [ ]:
calculator_spec = lab.ContractPolicySpec(
    name="fundamentals.calculator_agent.default_problem",
    description="Resolver el problema default con procedimiento y resultado final.",
    contract=lab.AgentContract(
        must_call=["sumar", "restar", "multiplicar", "dividir"],
        tool_expectation=lab.expect.all_of("sumar", "restar", "multiplicar", "dividir"),
        completion="when_required_tools_satisfied",
        failure_policy="no_unresolved",
        expected_tool_outputs={
            "sumar": {"result": 30},
            "restar": {"result": 21},
            "multiplicar": {"result": 84},
            "dividir": {"result": 42},
        },
    ),
    policy=lab.RunPolicy(max_turns=8, max_tool_calls=4, temperature=0.0, finalize="after_required_tools"),
)

lab.show({
    "spec": calculator_spec.describe(),
    "static_check": calculator_spec.check(available_tools=[tool.name for tool in tools]).to_dict(),
})

## 3) Crear un agente determinista con `python-direct`

Este agente no usa LLM. El runtime local ejecuta un plan determinista contra las tools registradas. Sirve para pruebas unitarias, contratos y ejemplos que deben correr igual en cualquier m?quina.

In [ ]:
instructions = """
Eres un agente calculadora.
Usa las tools disponibles para resolver el problema aritm?tico.
No hagas c?lculo mental cuando exista una tool adecuada.
Responde en espa?ol con:
- procedimiento
- resultado final
""".strip()

agent_controls = calculator_spec.agent_kwargs()

deterministic_agent = lab.agent(
    name="calculator_deterministic_agent",
    instructions=instructions,
    tools=tools,
    engine="python-direct",
    runtime=deterministic_runtime,
    **agent_controls,
)

lab.show({
    "agent": deterministic_agent.info(),
    "contract_policy": calculator_spec.describe(),
    "agent_controls": agent_controls,
}, title="Agente determinista")

## 4) Ejecutar el agente determinista

La entrada es estructurada para que `python-direct` pueda demostrar el contrato sin depender de interpretaci?n de lenguaje natural.

In [ ]:
deterministic_input = {"tool": "sumar", "input": {"a": 10, "b": 20}}

single_call_contract = lab.AgentContract(
    must_call=["sumar"],
    tool_expectation=lab.expect.exactly("sumar"),
    completion="when_required_tools_satisfied",
)

deterministic_agent.contract = single_call_contract

deterministic_result = deterministic_agent.run(deterministic_input, mode="eval")
deterministic_result.validation = deterministic_result.validate(single_call_contract).to_dict()

lab.human_result(
    deterministic_result,
    title="Human result ? agente determinista ? python-direct",
    expected_tools=single_call_contract.tool_expectation,
    pretty=PRETTY,
)

## 5) Crear un agente LM con `runtime(provider="auto")`

`auto` no significa magia oculta: `RuntimeConfig.describe()` declara qu? provider seleccion? por se?ales del ambiente.

- En VS Code local o sandbox, `auto` selecciona el backend disponible por se?ales del ambiente.
- Si no hay se?ales, el notebook salta esta ejecuci?n y deja expl?cita la raz?n.
- El c?digo del agente no cambia cuando cambias de backend.


In [ ]:
lm_resolution = lm_runtime.describe()
lm_provider = lm_resolution["selected_provider"]

lm_workspace = lab.AgenticSystem(
    model=lm_runtime.model_id or lab.default_model_id(),
    region=lm_runtime.region_name or lab.default_region(),
    runtime=lm_runtime,
)

lm_agent = lm_workspace.agent(
    name="calculator_lm_auto_agent",
    instructions=instructions,
    tools=tools,
    runtime=lm_runtime,
    contract=calculator_spec.contract,
    policy=calculator_spec.policy,
)

lab.show({
    "auto_resolution": lm_resolution,
    "lm_agent": lm_agent.info(),
}, title="Agente LM con provider auto")


## 6) Ejecutar el agente LM si hay provider disponible

Esta celda mantiene el tutorial agn?stico: no fuerza OpenAI ni Bedrock. Si `auto` no encuentra configuraci?n, se reporta como skip controlado.


In [ ]:
if lm_provider == "auto":
    lab.show({
        "status": "skipped",
        "reason": lm_resolution["reason"],
        "how_to_enable": "Configura OPENAI_API_KEY o credenciales/configuraci?n Bedrock antes de abrir el kernel.",
    }, title="Agente LM saltado")
else:
    lm_result = lm_agent.run(USER_PROMPT, mode="eval")
    lm_result.validation = lm_result.validate(calculator_spec.contract).to_dict()

    lab.human_result(
        lm_result,
        title=f"Human result ? agente LM ? {lm_provider}",
        expected_tools=calculator_spec.contract.tool_expectation,
        pretty=PRETTY,
    )


## Lo importante

- `lab.agent(...)` no oculta el contrato.
- `engine="python-direct"` es la ruta determinista local.
- `runtime(provider="auto")` es la ruta LM agn?stica al backend.
- `AgentContract` define las tools esperadas.
- `RunPolicy` controla presupuesto y finalizaci?n.
- `RunResult.validate(...)` separa ejecuci?n de validaci?n.
- `human_result(...)` renderiza, no decide la verdad del dominio.

## Coverage API de este notebook

Esta tabla deja explícito qué parte de Agentic Systems queda materializada aquí.

In [ ]:
api_coverage = [
    {
        "api": "lab.agent",
        "description": "Crea agentes canonicos con contrato, tools y runtime declarados."
    },
    {
        "api": 'engine="python-direct"',
        "description": "Ejecuta agentes deterministas locales sin proveedor externo."
    },
    {
        "api": 'runtime(provider="auto")',
        "description": "Ejecuta agentes LM con seleccion automatica de provider segun el ambiente."
    },
    {
        "api": "AgenticSystem(...).agent",
        "description": "Vincula un agente LM a un workspace que puede resolver providers reales."
    },
    {
        "api": "AgentContract",
        "description": "Define que tools y salidas son obligatorias para el agente."
    },
    {
        "api": "RunPolicy",
        "description": "Controla el comportamiento de ejecucion del agente."
    },
    {
        "api": "RunResult.validate",
        "description": "Valida el resultado contra el contrato despues de ejecutar."
    },
    {
        "api": "human_result",
        "description": "Renderiza la salida humana sin perder la evidencia interna."
    },
    {
        "api": "default arithmetic prompt",
        "description": "Usa el mismo prompt base para comparar comportamiento entre notebooks."
    }
]

lab.show({'notebook': '03_agent_api.ipynb', 'api_coverage': api_coverage})